In [ ]:
from IPython.display import Markdown, display
# Initialization, styling configuration, and helper utilities
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np
import torch
from ipywidgets import interact

# Load course helper utilities (local file or direct GitHub fetch for standalone Colab)
try:
    import helpers as hp
except ImportError:
    import urllib.request
    _url = "https://raw.githubusercontent.com/konstantin-schekotihin/dl_course/master/ML-DL/02_deep_learning/helpers.py"
    urllib.request.urlretrieve(_url, "helpers.py")
    import helpers as hp

import importlib
importlib.reload(hp)

hp.setup_theme()

# Single-layer Networks: Classification
## Linear Discriminants, Decision Theory, and Logistic Models

This lecture explores classification through single-layer architectures and decision boundaries:
- **Linear Discriminant Functions**: Hyperplane geometry, normal vectors, and signed distances
- **Multi-Class Discriminants**: Convex decision regions and 1-of-$K$ target coding
- **The Perceptron Algorithm**: Rosenblatt's continuous error criterion and the Novikoff convergence theorem
- **The Limits of Linear Separability**: The XOR crisis and the historical necessity for multi-layer networks
- **Least Squares for Classification**: Why squared error fails for discrete classification
- **Decision Theory**: Expected loss, loss matrices, and the reject option
- **Logistic Regression**: Bernoulli likelihood, cross-entropy loss, and IRLS optimization
- **Multi-Class Logistic Regression**: The softmax activation function and categorical cross-entropy

# Linear Discriminant Functions
## Geometry of the Separating Hyperplane

A **linear discriminant** takes an input vector $\mathbf{x}$ and assigns it to a discrete class by evaluating a linear combination of features:
$$y(\mathbf{x}) = \mathbf{w}^\mathrm{T}\mathbf{x} + w_0$$
where $\mathbf{w}$ is the weight vector and $w_0$ is the bias (or threshold parameter).

For a two-class problem with classes $\mathcal{C}_1$ and $\mathcal{C}_2$:
- Assign $\mathbf{x} \in \mathcal{C}_1$ if $y(\mathbf{x}) \ge 0$
- Assign $\mathbf{x} \in \mathcal{C}_2$ if $y(\mathbf{x}) < 0$

The decision boundary is defined by the set of points satisfying:
$$y(\mathbf{x}) = \mathbf{w}^\mathrm{T}\mathbf{x} + w_0 = 0$$

This defines a $(D-1)$-dimensional **hyperplane** within the $D$-dimensional input space.

## Geometric Properties of the Decision Boundary

The parameters $\mathbf{w}$ and $w_0$ govern the orientation and position of the hyperplane:

1. **Orientation**:
   Take any two points $\mathbf{x}_A$ and $\mathbf{x}_B$ lying on the decision surface ($y(\mathbf{x}_A) = y(\mathbf{x}_B) = 0$):
   $$\mathbf{w}^\mathrm{T}\mathbf{x}_A + w_0 = 0 \quad \text{and} \quad \mathbf{w}^\mathrm{T}\mathbf{x}_B + w_0 = 0 \implies \mathbf{w}^\mathrm{T}(\mathbf{x}_A - \mathbf{x}_B) = 0$$

   > 💡 **Geometric Rule:** The weight vector $\mathbf{w}$ is strictly **orthogonal to every vector lying within the decision surface**. It defines the normal vector to the hyperplane.

2. **Distance from the Origin**:
   For any point $\mathbf{x}$ on the decision surface, its projection onto $\mathbf{w}$ gives the orthogonal distance from the origin:
   $$\frac{\mathbf{w}^\mathrm{T}\mathbf{x}}{\|\mathbf{w}\|} = -\frac{w_0}{\|\mathbf{w}\|}$$

3. **Signed Distance of an Arbitrary Point**:
   Any point $\mathbf{x}$ can be decomposed into its orthogonal projection $\mathbf{x}_\perp$ onto the surface plus a normal displacement:
   $$\mathbf{x} = \mathbf{x}_\perp + r \frac{\mathbf{w}}{\|\mathbf{w}\|} \implies r = \frac{y(\mathbf{x})}{\|\mathbf{w}\|}$$

## Multi-Class Discriminants & Convexity

Generalizing two-class discriminants to $K > 2$ classes using multiple binary classifiers introduces severe ambiguities:
- **One-versus-the-rest**: Requires $K$ classifiers; leads to ambiguous regions where points are claimed by multiple classes or by none.
- **One-versus-one**: Requires $K(K-1)/2$ pairwise classifiers; leads to ambiguous voting cycles.

The principled solution is a **$K$-class linear discriminant** comprising $K$ linear functions:
$$y_k(\mathbf{x}) = \mathbf{w}_k^\mathrm{T}\mathbf{x} + w_{k0}, \quad k = 1, \dots, K$$

A point $\mathbf{x}$ is assigned to class $\mathcal{C}_k$ if:
$$y_k(\mathbf{x}) > y_j(\mathbf{x}) \quad \forall j \neq k$$

The decision boundary between class $\mathcal{C}_k$ and $\mathcal{C}_j$ is given by $y_k(\mathbf{x}) = y_j(\mathbf{x})$, which corresponds to a linear hyperplane:
$$(\mathbf{w}_k - \mathbf{w}_j)^\mathrm{T}\mathbf{x} + (w_{k0} - w_{j0}) = 0$$

> 📘 **Convexity Theorem:** Because each decision region $\mathcal{R}_k$ is defined by the intersection of open half-spaces ($y_k(\mathbf{x}) > y_j(\mathbf{x})$), **every decision region $\mathcal{R}_k$ is singly connected and convex**.

# The Rosenblatt Perceptron
## Historical Roots of Connectionist Learning

Invented by Frank Rosenblatt in 1958 at Cornell Aeronautical Laboratory, the **Perceptron** represents the earliest computational model of supervised learning.

The Perceptron transforms inputs $\mathbf{x}$ through fixed non-linear basis functions $\boldsymbol{\phi}(\mathbf{x})$ and applies a step/sign activation function:
$$y(\mathbf{x}) = f\left(\mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})\right)$$
where the non-linear activation function $f(a)$ is a strict threshold step function:
$$f(a) = \begin{cases} +1, & a \ge 0 \\ -1, & a < 0 \end{cases}$$

Target variables are coded as $t_n \in \{-1, +1\}$. A sample $\mathbf{x}_n$ is correctly classified if:
$$\mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n) t_n > 0$$

<small>📖 *Source: [Rosenblatt, F. (1958). The perceptron: A probabilistic model for information storage and organization in the brain. Psychological Review, 65(6), 386–408](https://doi.org/10.1037/h0042519)*</small>

## The Perceptron Error Criterion

Why not directly minimize the count of misclassified samples?
The number of misclassified samples is piecewise constant: its gradient is zero almost everywhere and undefined at the boundaries, rendering gradient-based optimization useless.

To formulate a smooth gradient-amenable loss, Rosenblatt introduced the **Perceptron criterion**:
- For any misclassified point $\mathbf{x}_n \in \mathcal{M}$, we have $\mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n t_n < 0$.
- The contribution to the error is defined as $-\mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n t_n > 0$.
- Correctly classified points contribute zero to the error.

Summing over the set of currently misclassified samples $\mathcal{M}$:
$$E_\mathrm{P}(\mathbf{w}) = -\sum_{n \in \mathcal{M}} \mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x}_n) t_n$$

## The Perceptron Learning Rule

The gradient with respect to $\mathbf{w}$ on an individual misclassified sample $n$ is:
$$\nabla E_{\mathrm{P}, n}(\mathbf{w}) = -\boldsymbol{\phi}(\mathbf{x}_n) t_n$$

Applying Stochastic Gradient Descent yields the celebrated **Perceptron Learning Rule**:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \eta \nabla E_{\mathrm{P}, n}(\mathbf{w}) = \mathbf{w}^{(\tau)} + \eta \boldsymbol{\phi}(\mathbf{x}_n) t_n$$
where $\eta > 0$ is the learning rate (without loss of generality, $\eta = 1$).

```
Geometric Interpretation of the Weight Update:
If misclassified t_n = +1:  w is rotated towards phi(x_n).
If misclassified t_n = -1:  w is rotated away from phi(x_n).
```

## Novikoff's Perceptron Convergence Theorem

Does the Perceptron learning algorithm always find a separating hyperplane?

> 📘 **Novikoff's Theorem (1962):**
> If the training dataset $\mathcal{D} = \{(\mathbf{x}_n, t_n)\}_{n=1}^N$ is **linearly separable** in feature space $\boldsymbol{\phi}(\mathbf{x})$, then starting from initial weights $\mathbf{w}^{(0)} = \mathbf{0}$, the Perceptron learning algorithm is guaranteed to converge to an exact separating hyperplane in a **finite number of steps** $\tau$:

$$\tau \le \left(\frac{R}{\gamma}\right)^2$$

where $R = \max_n \|\boldsymbol{\phi}(\mathbf{x}_n)\|$ is the maximum data radius, and $\gamma = \min_n \frac{\mathbf{w}^{*\mathrm{T}}\boldsymbol{\phi}_n t_n}{\|\mathbf{w}^*\|} > 0$ is the **geometric margin** separating the classes.

<small>📖 *Source: [Novikoff, A. B. J. (1962). On convergence proofs for perceptrons. Symposium on the Mathematical Theory of Automata, 12, 615–622](https://archive.org/details/nasa_techdoc_19630006730)*</small>

> ⚠️ **The Linearity Limitation:** If the data is **not** linearly separable, the algorithm does not find an approximate solution: it **cycles indefinitely** without terminating.

In [ ]:
# Implementation of the Rosenblatt Perceptron Class
class Perceptron:
    def __init__(self, lr=1.0, max_epochs=100):
        self.lr, self.max_epochs = lr, max_epochs
        self.w, self.converged, self.epochs_run = None, False, 0

    def fit(self, X, t):
        X_b = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
        N, D = X_b.shape
        self.w = torch.zeros(D, 1)

        for epoch in range(self.max_epochs):
            misclassified = 0
            for n in range(N):
                x_n, t_n = X_b[n:n+1].T, t[n].item()
                if torch.matmul(self.w.T, x_n).item() * t_n <= 0:
                    self.w += self.lr * t_n * x_n
                    misclassified += 1

            if misclassified == 0:
                self.converged, self.epochs_run = True, epoch + 1
                return self

        self.epochs_run = self.max_epochs
        return self

    def predict(self, X):
        X_b = torch.cat([torch.ones(X.shape[0], 1), X], dim=1)
        return torch.where(torch.matmul(X_b, self.w) >= 0, 1.0, -1.0)

In [ ]:
# Evaluating Perceptron on AND, OR, and XOR Logic Gates
logic_data = {
    'and': (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]),
            torch.tensor([[-1.], [1.], [-1.], [-1.]])),
    'or':  (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]),
            torch.tensor([[-1.], [1.], [1.], [1.]])),
    'xor': (torch.tensor([[0., 0.], [1., 1.], [1., 0.], [0., 1.]]),
            torch.tensor([[-1.], [-1.], [1.], [1.]]))
}

print("=== PERCEPTRON TRAINING ON LOGIC GATES ===")
for gate_name in ['and', 'or', 'xor']:
    X_gate, t_gate = logic_data[gate_name]
    p_model = Perceptron(lr=0.5, max_epochs=50).fit(X_gate, t_gate)
    preds = p_model.predict(X_gate).squeeze().numpy()
    status = f"Converged in {p_model.epochs_run} epochs" if p_model.converged else "FAILED to converge (Non-separable)"
    print(f"Gate {gate_name.upper():3s} -> {status} | Weights: {p_model.w.numpy().ravel().round(2)} | Preds: {preds}")

## The XOR Crisis & The First AI Winter

In their landmark 1969 book *Perceptrons*, Marvin Minsky and Seymour Papert proved mathematically that single-layer perceptrons are incapable of learning non-linearly separable functions such as **XOR**:

```
  x_2 ^
      |  (0,1) [+]         (1,1) [-]
      |
      |  (0,0) [-]         (1,0) [+]
      +------------------------------> x_1
```
No single linear line can separate the `[+]` points from the `[-]` points!

<small>📖 *Source: [Minsky, M., & Papert, S. (1969). Perceptrons: An Introduction to Computational Geometry. MIT Press](https://mitpress.mit.edu/9780262631112/perceptrons/)*</small>

> 💡 **Historical Consequence:** Minsky and Papert's result demonstrated that single-layer linear networks are fundamentally limited, leading to a decade-long funding collapse ("AI Winter"). Resolving this crisis required **Multi-Layer Perceptrons (MLPs)** with hidden representation layers, trained via backpropagation.

In [ ]:
# Standalone Interactive Widget Helper for Perceptron Decision Boundaries
def test_perceptron_gate(gate='and'):
    X_gate, t_gate = logic_data[gate]
    p = Perceptron(lr=0.2, max_epochs=50).fit(X_gate, t_gate)
    y_binary = (t_gate.squeeze() > 0).numpy().astype(int)
    
    plt.figure(figsize=(6, 4))
    x_grid = np.linspace(-0.5, 1.5, 100)
    xx, yy = np.meshgrid(x_grid, x_grid)
    grid_tensor = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    grid_preds = p.predict(grid_tensor).reshape(xx.shape).numpy()
    
    plt.contourf(xx, yy, grid_preds, alpha=0.3, cmap='coolwarm')
    plt.scatter(X_gate[:, 0].numpy(), X_gate[:, 1].numpy(), 
                c=y_binary, cmap='coolwarm', s=120, edgecolors='k', lw=2)
    plt.title(f"Perceptron Decision Boundary: {gate.upper()} (Converged: {p.converged})", fontsize=12)
    plt.xlabel("x_1")
    plt.ylabel("x_2")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

In [ ]:
# Interactive exploration of Perceptron linear separability across logic gates
interact(test_perceptron_gate, gate=['and', 'or', 'xor']);

# Least Squares for Classification
## Why Squared Error Fails for Discrete Targets

It is tempting to treat classification as regression: code targets as $\mathbf{t}_n \in \{0, 1\}$ or 1-of-$K$ vectors and minimize squared error $\|\mathbf{T} - \mathbf{X}\mathbf{W}\|^2$.

However, least squares exhibits two severe pathological failure modes when applied to classification:

1. **Extreme Sensitivity to Outliers**:
   - Points that are correctly classified but lie far from the boundary receive a **huge penalty** under $(y - 1)^2$.
   - The optimization shifts the decision boundary to reduce these artificial errors on correctly classified points, which causes points close to the boundary to be misclassified!

2. **The Masking Problem**:
   - In multi-class problems ($K \ge 3$), classes whose instances lie between other classes can be completely "masked" (subsumed) by outer classes, with the model assigning zero instances to the intermediate class.

> ⚠️ **Rule:** Squared error measures Euclidean distance to target values, not decision correctness or posterior class probabilities.

In [ ]:
# Demonstration of Least Squares Sensitivity to Distant Outliers
np.random.seed(42)
c1 = np.random.randn(20, 2) * 0.4 + np.array([-1.0, 0.0])
c2 = np.random.randn(20, 2) * 0.4 + np.array([+1.0, 0.0])

# Add distant outliers belonging to Class 2 far to the right
outliers = np.random.randn(10, 2) * 0.4 + np.array([+6.0, 0.0])

X_clean = np.vstack([c1, c2])
t_clean = np.array([-1]*20 + [+1]*20)

X_outlier = np.vstack([c1, c2, outliers])
t_outlier = np.array([-1]*20 + [+1]*20 + [+1]*10)

def fit_ls(X, t):
    X_b = np.hstack([np.ones((X.shape[0], 1)), X]
    )
    return np.linalg.pinv(X_b) @ t

w_clean = fit_ls(X_clean, t_clean)
w_outlier = fit_ls(X_outlier, t_outlier)

display(Markdown(
    r"**Sensitivity of Least Squares to Outliers**:"
    + f"\n- Clean Data Weights $\\mathbf{{w}} = [w_0, w_1, w_2]^\\mathrm{{T}}$: `{w_clean.round(3).tolist()}`"
    + f"\n- Outlier Data Weights $\\mathbf{{w}} = [w_0, w_1, w_2]^\\mathrm{{T}}$: `{w_outlier.round(3).tolist()}`"
    + "\n\n*Notice the shift in decision boundary weights caused by points that are already correctly classified!*"
))


# Decision Theory for Classification
## Optimal Decisions & Loss Matrices

Classification separates naturally into two distinct stages:
1. **Inference Stage**: Using training data to estimate posterior class probabilities $p(\mathcal{C}_k \mid \mathbf{x})$.
2. **Decision Stage**: Using posterior probabilities to make optimal class assignments.

### 1. Minimizing the Misclassification Rate
When all errors are equally costly, the probability of mistake is minimized by assigning $\mathbf{x}$ to the class with the largest posterior probability:
$$\hat{k} = \arg\max_k p(\mathcal{C}_k \mid \mathbf{x})$$

### 2. Minimizing Expected Loss via Loss Matrices
In medical diagnosis, classifying a sick patient as healthy ($L_{12}$) is catastrophic, whereas classifying a healthy patient as sick ($L_{21}$) causes minor re-testing cost ($L_{12} \gg L_{21}$).
We quantify consequences via the **loss matrix** $L_{kj}$ (cost of assigning a true class $\mathcal{C}_k$ item to class $\mathcal{C}_j$).

The expected loss is minimized by choosing decision regions $\mathcal{R}_j$ that satisfy:
$$\hat{j} = \arg\min_j \sum_{k=1}^K L_{kj} p(\mathcal{C}_k \mid \mathbf{x})$$

## The Reject Option in High-Stakes Decisions

When the largest posterior probability $\max_k p(\mathcal{C}_k \mid \mathbf{x})$ is close to $1/K$, classification confidence is low.
In safety-critical settings (e.g. autonomous driving, medical diagnostics), committing to an uncertain decision carries unacceptable risk.

We introduce a confidence threshold $\theta \in [1/K, 1)$:
- If $\max_k p(\mathcal{C}_k \mid \mathbf{x}) \ge \theta$: automate decision $\hat{k} = \arg\max_k p(\mathcal{C}_k \mid \mathbf{x})$.
- If $\max_k p(\mathcal{C}_k \mid \mathbf{x}) < \theta$: **reject** the automated decision and route to human expert.

> 💡 **Trade-off:** Varying threshold $\theta$ allows tuning the trade-off between the fraction of rejected cases and the error rate on non-rejected samples.

## Performance Metrics for Binary Classification

Given class predictions and ground truth, we tabulate the **Confusion Matrix**:

| | Predicted Positive ($\hat{t} = 1$) | Predicted Negative ($\hat{t} = 0$) |
| :--- | :---: | :---: |
| **Actual Positive ($t = 1$)** | True Positive (TP) | False Negative (FN) |
| **Actual Negative ($t = 0$)** | False Positive (FP) | True Negative (TN) |

- **Accuracy**: $\frac{\mathrm{TP} + \mathrm{TN}}{\mathrm{TP} + \mathrm{TN} + \mathrm{FP} + \mathrm{FN}}$
- **Precision**: $\frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FP}}$
- **Recall (Sensitivity)**: $\frac{\mathrm{TP}}{\mathrm{TP} + \mathrm{FN}}$
- **F1 Score**: $2 \cdot \frac{\mathrm{Precision} \cdot \mathrm{Recall}}{\mathrm{Precision} + \mathrm{Recall}}$
- **ROC-AUC**: Receiver Operating Characteristic curve; Area Under Curve measures threshold-independent ranking quality.

# Discriminative Models: Logistic Regression
## Modeling Posterior Probabilities Directly

Instead of modeling class-conditional densities $p(\mathbf{x} \mid \mathcal{C}_k)$ (generative approach), **logistic regression** directly models the posterior class probability:
$$p(\mathcal{C}_1 \mid \mathbf{x}) = y(\mathbf{x}) = \sigma\left(\mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})\right) = \frac{1}{1 + \exp\left(-\mathbf{w}^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})\right)}$$
and $p(\mathcal{C}_2 \mid \mathbf{x}) = 1 - p(\mathcal{C}_1 \mid \mathbf{x}) = 1 - y(\mathbf{x})$.

Here, $\sigma(a)$ is the **logistic sigmoid function**, which maps real-valued activation $a \in (-\infty, \infty)$ strictly into valid probabilities $(0, 1)$.

The derivative of the logistic sigmoid exhibits a fundamental algebraic property:
$$\frac{\mathrm{d}\sigma}{\mathrm{d}a} = \sigma(a)\left(1 - \sigma(a)\right)$$

Proof:
$$\frac{\mathrm{d}}{\mathrm{d}a} (1 + e^{-a})^{-1} = -(1 + e^{-a})^{-2} (-e^{-a}) = \frac{1}{1 + e^{-a}} \cdot \frac{e^{-a}}{1 + e^{-a}} = \sigma(a)(1 - \sigma(a))$$

## Maximum Likelihood & Cross-Entropy Error

For a dataset of i.i.d. observations $\{(\mathbf{x}_n, t_n)\}_{n=1}^N$ with binary target coding $t_n \in \{0, 1\}$, the likelihood function follows the **Bernoulli distribution**:
$$p(\mathbf{t} \mid \mathbf{w}) = \prod_{n=1}^N y_n^{t_n} (1 - y_n)^{1 - t_n}, \quad y_n = \sigma(\mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n)$$

Taking the negative logarithm gives the **Cross-Entropy Error Function**:
$$E(\mathbf{w}) = -\ln p(\mathbf{t} \mid \mathbf{w}) = -\sum_{n=1}^N \left[t_n \ln y_n + (1 - t_n)\ln(1 - y_n)\right]$$

Differentiating $E(\mathbf{w})$ with respect to $\mathbf{w}$ using the chain rule:
$$\nabla E(\mathbf{w}) = -\sum_{n=1}^N \left[\frac{t_n}{y_n} - \frac{1 - t_n}{1 - y_n}\right] \frac{\partial y_n}{\partial a_n} \boldsymbol{\phi}_n = -\sum_{n=1}^N \frac{t_n - y_n}{y_n(1 - y_n)} \cdot y_n(1 - y_n) \boldsymbol{\phi}_n$$

This simplifies to an astonishingly elegant result:
$$\nabla E(\mathbf{w}) = \sum_{n=1}^N \left(y_n - t_n\right) \boldsymbol{\phi}_n$$

> 💡 **Remarkable Structural Harmony:** The gradient of cross-entropy for logistic regression has the **exact same mathematical form** as the gradient of least squares for linear regression: $\sum (y_n - t_n)\boldsymbol{\phi}_n$ (error times feature vector)!

## Iterative Reweighted Least Squares (IRLS)

Because $y_n$ is a non-linear function of $\mathbf{w}$, setting $\nabla E(\mathbf{w}) = \mathbf{0}$ does not yield a closed-form analytical solution.
Instead, we apply the second-order **Newton-Raphson optimization method**:
$$\mathbf{w}^{(\tau+1)} = \mathbf{w}^{(\tau)} - \mathbf{H}^{-1} \nabla E(\mathbf{w})$$

The Hessian matrix $\mathbf{H} = \nabla^2 E(\mathbf{w})$ is obtained by differentiating $\nabla E(\mathbf{w})$:
$$\mathbf{H} = \nabla^2 E(\mathbf{w}) = \sum_{n=1}^N y_n(1 - y_n) \boldsymbol{\phi}_n \boldsymbol{\phi}_n^\mathrm{T} = \boldsymbol{\Phi}^\mathrm{T}\mathbf{R}\boldsymbol{\Phi}$$
where $\mathbf{R} \in \mathbb{R}^{N \times N}$ is a diagonal weighting matrix with elements $R_{nn} = y_n(1 - y_n) > 0$.

> 📘 **Strict Convexity:** Because $R_{nn} > 0$ for all $n$, the Hessian matrix $\mathbf{H} = \boldsymbol{\Phi}^\mathrm{T}\mathbf{R}\boldsymbol{\Phi}$ is **strictly positive-definite** whenever $\boldsymbol{\Phi}$ has full column rank. Therefore:
> - The cross-entropy error surface is **strictly convex**.
> - It contains **no local minima** — any local stationary point is the unique global minimum!

The Newton-Raphson update rewrites as weighted least squares (**IRLS**):
$$\mathbf{w}^{(\tau+1)} = \left(\boldsymbol{\Phi}^\mathrm{T}\mathbf{R}\boldsymbol{\Phi}\right)^{-1} \boldsymbol{\Phi}^\mathrm{T}\mathbf{R}\mathbf{z}, \quad \mathbf{z} = \boldsymbol{\Phi}\mathbf{w}^{(\tau)} - \mathbf{R}^{-1}(\mathbf{y} - \mathbf{t})$$

# Multi-Class Logistic Regression
## The Softmax Activation Function

For $K > 2$ classes, we generalize the logistic sigmoid to the **Softmax function**:
$$p(\mathcal{C}_k \mid \mathbf{x}) = y_k(\mathbf{x}) = \frac{\exp(a_k)}{\sum_{j=1}^K \exp(a_j)}, \quad a_k = \mathbf{w}_k^\mathrm{T}\boldsymbol{\phi}(\mathbf{x})$$

Softmax guarantees that:
1. $y_k(\mathbf{x}) \in (0, 1)$ for all $k$.
2. $\sum_{k=1}^K y_k(\mathbf{x}) = 1$.

Representing class targets using 1-of-$K$ coding $\mathbf{t}_n \in \{0, 1\}^K$, the likelihood follows the **multinomial distribution**:
$$p(\mathbf{T} \mid \mathbf{W}) = \prod_{n=1}^N \prod_{k=1}^K y_{nk}^{t_{nk}}$$

The multi-class cross-entropy error function is:
$$E(\mathbf{W}) = -\ln p(\mathbf{T} \mid \mathbf{W}) = -\sum_{n=1}^N \sum_{k=1}^K t_{nk} \ln y_{nk}$$

The gradient with respect to parameter vector $\mathbf{w}_j$ preserves the universal error-times-feature form:
$$\nabla_{\mathbf{w}_j} E(\mathbf{W}) = \sum_{n=1}^N \left(y_{nj} - t_{nj}\right) \boldsymbol{\phi}_n$$

In [ ]:
# PyTorch Implementation of Binary Logistic Regression
class LogisticRegression(torch.nn.Module):
    def __init__(self, in_features):
        super().__init__()
        self.linear = torch.nn.Linear(in_features, 1)

    def forward(self, x):
        return torch.sigmoid(self.linear(x))

# Generate synthetic 2D linearly separable clusters
torch.manual_seed(42)
N_pts = 50
X_0 = torch.randn(N_pts, 2) * 0.7 + torch.tensor([-1.5, -1.0])
X_1 = torch.randn(N_pts, 2) * 0.7 + torch.tensor([+1.5, +1.0])
X_toy = torch.cat([X_0, X_1], dim=0)
t_toy = torch.cat([torch.zeros(N_pts, 1), torch.ones(N_pts, 1)], dim=0)

model_lr = LogisticRegression(in_features=2)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model_lr.parameters(), lr=0.1)

# Training loop
for epoch in range(200):
    optimizer.zero_grad()
    loss = criterion(model_lr(X_toy), t_toy)
    loss.backward()
    optimizer.step()

print(f"Final Binary Cross-Entropy Loss: {loss.item():.4f}")
print("Trained Weights:", model_lr.linear.weight.detach().numpy().ravel().round(3))
print("Trained Bias   :", model_lr.linear.bias.detach().numpy().round(3))

In [ ]:
# Decision Boundary Visualization via helpers
def predictor_fn(x_np):
    with torch.no_grad():
        x_t = torch.tensor(x_np, dtype=torch.float32)
        return (model_lr(x_t).squeeze() > 0.5).numpy().astype(float)

hp.plot_binary(predictor_fn, X_toy.numpy(), t_toy.squeeze().numpy())

# Summary: Linear Models & Perceptrons

In this lecture, we explored linear discriminants and classification models:

1. **Linear Discriminant Functions**:
   $$y(\mathbf{x}) = \mathbf{w}^\mathrm{T}\mathbf{x} + w_0$$
   $\mathbf{w}$ represents the normal vector governing hyperplane orientation; $w_0$ dictates displacement from the origin.

2. **The Perceptron Algorithm**:
   - Rosenblatt's continuous piecewise-linear criterion: $E_\mathrm{P}(\mathbf{w}) = -\sum_{n \in \mathcal{M}} \mathbf{w}^\mathrm{T}\boldsymbol{\phi}_n t_n$.
   - **Novikoff's Theorem**: Guaranteed finite convergence for linearly separable data.
   - **The XOR Crisis**: Inability of single-layer perceptrons to separate non-linear configurations.

## Summary: Probabilistic Discriminative Models

3. **Least Squares Failure for Classification**:
   Extreme sensitivity to distant outliers and class masking make squared error unsuitable for classification.

4. **Logistic Regression & Softmax**:
   - Directly models posterior class probabilities using logistic sigmoid (binary) or softmax (multi-class).
   - Cross-entropy loss yields strictly convex error surfaces with unique global minima.
   - Gradient preserves universal form: $\nabla E(\mathbf{w}) = \sum (y_n - t_n)\boldsymbol{\phi}_n$.